In [1]:
import pandas as pd
import numpy as np
import re
import string

# Load dataset (adjust filename and separator if yours is a .txt or differently named .csv)
# Note: CodSoft dataset is often tab/pipe separated or a CSV
try:
    df = pd.read_csv('train_data.txt', sep=':::', engine='python', names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION'])
except Exception:
    df = pd.read_csv('wiki_movie_plots_deduped.csv')

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Dataset loaded successfully!
Shape: (54214, 4)

First 5 rows:


,ID,TITLE,GENRE,DESCRIPTION
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...


In [2]:
import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()                                    # Lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)    # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)               # Remove special characters & numbers
    words = text.split()
    words = [w for w in words if w not in stop_words]     # Remove stop words
    return " ".join(words)

# Apply text cleaning to movie descriptions
df['CLEAN_DESCRIPTION'] = df['DESCRIPTION'].apply(clean_text)

print("\nSample Original Description:")
print(df['DESCRIPTION'].iloc[0][:150])
print("\nSample Cleaned Description:")
print(df['CLEAN_DESCRIPTION'].iloc[0][:150])

print("\nGenre Distribution (Top 10):")
print(df['GENRE'].value_counts().head(10))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.



Sample Original Description:
 Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few we

Sample Cleaned Description:
listening conversation doctor parents yearold oscar learns nobody courage tell weeks live furious refuses speak anyone except straighttalking rose lad

Genre Distribution (Top 10):
GENRE
drama           13613
documentary     13096
comedy           7447
short            5073
horror           2204
thriller         1591
action           1315
western          1032
reality-tv        884
family            784
Name: count, dtype: int64


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Limit features to top 5000 terms for speed and memory efficiency
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['CLEAN_DESCRIPTION'])
y = df['GENRE']

print("TF-IDF Matrix Shape:", X.shape)

TF-IDF Matrix Shape: (54214, 5000)


In [4]:
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 43371
Testing samples: 10843


In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Train Multinomial Naive Bayes model
model = MultinomialNB()
model.fit(X_train, y_train)

print("Text Classification model trained successfully!")

Text Classification model trained successfully!


In [6]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on test set
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"✅ Model Accuracy: {acc * 100:.2f}%\n")

print("Classification Report (Top Genres):")
print(classification_report(y_test, y_pred, zero_division=0))

✅ Model Accuracy: 52.24%

Classification Report (Top Genres):
               precision    recall  f1-score   support

      action        0.59      0.06      0.12       263
       adult        0.62      0.04      0.08       118
   adventure        0.80      0.05      0.10       155
   animation        0.00      0.00      0.00       100
   biography        0.00      0.00      0.00        53
      comedy        0.51      0.43      0.47      1490
       crime        0.00      0.00      0.00       101
 documentary        0.57      0.90      0.70      2619
       drama        0.46      0.82      0.59      2723
      family        0.00      0.00      0.00       157
     fantasy        0.00      0.00      0.00        65
   game-show        1.00      0.18      0.30        39
     history        0.00      0.00      0.00        49
      horror        0.75      0.36      0.48       441
       music        0.78      0.10      0.17       146
     musical        0.00      0.00      0.00        55
  

In [7]:
def predict_genre(description):
    cleaned = clean_text(description)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)
    return prediction[0]

# Custom test examples
sample_1 = "A brave detective travels through dark alleys to fight corrupt cops and uncover a conspiracy."
sample_2 = "Two young lovers meet on a cruise ship, facing social class struggles and tragic events."

print("Sample 1 Predicted Genre:", predict_genre(sample_1))
print("Sample 2 Predicted Genre:", predict_genre(sample_2))

Sample 1 Predicted Genre:  action 
Sample 2 Predicted Genre:  drama 
